# `ptof_obs_behavioral_correlation.ipynb`

## What this notebook does
Correlates AI-generated shift-handover output against downstream human/ISH activity to catch
failure modes that live outside the LLM call itself: handovers that silently failed to send, and
(when identity resolution allows it) human corrections made shortly after an AI publish. These are
behavioral signals — they require joining LLM bronze events to ISH system-of-record events on
shift/batch identity, which is why `ish_entity_dim` exists as a dimension table in this notebook
rather than being computed inline.

## Position in the pipeline
- **Job:** `obs_fresh_scan` task `05_behavioral_correlation`, runs in parallel with
  `02`/`03`/`04` after `01_bronze_projections`.
- **Upstream:** reads `v_llm_bronze`, `v_ish_bronze`, `capability_registry` (all from
  `ptof_obs_bronze_projection.ipynb`), and builds/reads its own `ish_entity_dim`.
- **Downstream:** `ptof_obs_alert.ipynb` reads `handover_delivery_failures` (detector
  `handover_delivery`, CRITICAL) and `handover_delivery_rate` / `handover_delivery_rate_findings`
  (detector `handover_delivery_rate`, CRITICAL — the deterioration/rate-based counterpart).
  `rapid_human_correction` is **not** wired into alerting for v1: it always returns 0 rows because
  `dsa_*` capabilities never populate shift/batch identity on the LLM side, per Phase 4 item 10 of
  `agent_obs_implementation_plan.md` (registry/identity drift — still open).

## Tables/views touched
- **Reads:** `v_ish_bronze`, `v_llm_bronze`, `capability_registry`, `ish_entity_dim` (self-built).
- **Writes:** `ish_entity_dim`, `rapid_human_correction`, `handover_delivery_failures`,
  `handover_delivery_rate`, `handover_delivery_rate_findings`.


In [0]:
%sql
-- ish_entity_dim — resolves shift/batch for UPDATE and DELETE rows.
--
-- Necessary because after_json shape depends on ACTION, not just entity_type:
--   CREATE Note        -> {shift_date_key, shift_type, batch_id, ...}   keys present
--   CREATE ManualDowntime -> {shift_date, shift_type, batch_id, ...}    keys present
--   SEND   HandoverEmail  -> {shift_date, shift_label, content.batch_id} keys present
--   UPDATE Note        -> {text, is_pinned, type}                       NO keys
--   UPDATE Checklist   -> {item_key, done, completed_by, completed_at}  NO keys
--   DELETE (any)       -> after_json IS NULL entirely                   NO keys
--
-- So the shift/batch of a mutation has to come from the entity's own creation event, joined on
-- entity_id. Verified 2026-08-19: 100% resolution, zero entity_id collisions across Note,
-- ManualDowntime and DowntimeSplit (243 mutations).
--
-- Checklist and Config are EXCLUDED: they have no CREATE event at all (0% resolution), because
-- checklist items are a fixed template toggled in place rather than created per shift.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.ish_entity_dim AS
SELECT
    entity_id,
    entity_type,
    min_by(shift_date_norm, ts)  AS shift_date,
    min_by(shift_type_norm,  ts) AS shift_type_raw,
    min_by(batch_id_norm,    ts) AS batch_nbr,
    min(ts)                      AS created_at,
    min_by(user_email, ts)       AS created_by,
    count(*)                     AS n_create_events
FROM mq_gmdf_dev.oil_obs.v_ish_bronze
WHERE action IN ('CREATE','SEND')
  AND entity_type IN ('Note','HandoverEmail','ManualDowntime','DowntimeSplit')
  AND entity_id IS NOT NULL AND entity_id <> ''
  AND batch_id_norm IS NOT NULL
GROUP BY entity_id, entity_type;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- rapid_human_correction — human edits/deletes within 10 min of an AI publish, resolved through
-- ish_entity_dim. Formerly returned 0 rows: it joined ISH UPDATE/DELETE rows directly on
-- batch_id_norm, which is NULL on every such row.
--
-- Three normalizations, all learned from the data:
--   1. shift_type: HandoverEmail carries shift_label ('Night shift'); Note and ManualDowntime
--      carry shift_type ('night'). Strip the ' shift' suffix and lowercase both sides.
--   2. Exclude auto-handover@system so machine writes are not scored as human intervention.
--   3. Exclude is_email_disabled_gate rows (EMAIL_ENABLED=false is a config gate, not a failure).
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.rapid_human_correction AS
WITH ai_publish AS (
  SELECT b.id AS ai_row_id, b.capability, b.called_at AS ai_called_at,
         CAST(b.shift_date AS DATE)  AS shift_date,
         lower(trim(b.shift_type))   AS shift_type,
         b.batch_nbr
  FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
  JOIN mq_gmdf_dev.oil_obs.capability_registry r
    ON r.capability = b.capability AND r.active = true AND r.is_generative = true
  WHERE b.success = true
    AND b.is_blank_output = false
    AND b.is_credential_fastfail = false
    AND b.shift_type <> ''
    AND b.batch_nbr IS NOT NULL
    AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
),
ish_correction AS (
  SELECT i.id AS ish_row_id, i.entity_type, i.action, i.user_email, i.ts, i.change_summary,
         d.shift_date,
         lower(regexp_replace(trim(d.shift_type_raw), '(?i)\\s*shift\\s*$', '')) AS shift_type,
         d.batch_nbr, d.created_by
  FROM mq_gmdf_dev.oil_obs.v_ish_bronze i
  JOIN mq_gmdf_dev.oil_obs.ish_entity_dim d
    ON d.entity_id = i.entity_id AND d.entity_type = i.entity_type
  WHERE i.action IN ('UPDATE','DELETE')
    AND i.is_email_disabled_gate = false
    AND coalesce(i.user_email, '') <> 'auto-handover@system'
    AND i.ts >= current_timestamp() - INTERVAL 7 DAYS
)
SELECT a.ai_row_id, i.ish_row_id, a.capability,
       a.shift_date, a.shift_type, a.batch_nbr,
       a.ai_called_at, i.ts AS correction_ts,
       unix_timestamp(i.ts) - unix_timestamp(a.ai_called_at) AS time_to_correction_s,
       i.entity_type, i.action, i.user_email, i.created_by AS entity_created_by,
       i.change_summary,
       current_timestamp() AS detected_at
FROM ai_publish a
JOIN ish_correction i
  ON  i.shift_date = a.shift_date
  AND i.shift_type = a.shift_type
  AND i.batch_nbr  = a.batch_nbr
  AND i.ts BETWEEN a.ai_called_at AND a.ai_called_at + INTERVAL 10 MINUTES;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- handover_delivery_failures — a handover that silently failed to send.
-- Two sent=false reasons exist in this data:
--   'email disabled in this environment (EMAIL_ENABLED=false)' -> config gate, excluded
--   'SMTP send failed: [Errno 11001] getaddrinfo failed'       -> real delivery failure
-- In a shift-handover system a handover that never reached the distribution list is arguably a
-- worse defect than a mildly ungrounded number, and nothing detected it before.
-- recipients/subject are masked (local-part before @ replaced with ***, subject bounded to
-- 200 chars) -- both may contain real distribution-list addresses / free text, never stored raw.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.handover_delivery_failures AS
SELECT
    i.id                              AS ish_row_id,
    i.ts                              AS attempted_at,
    i.shift_date_norm                 AS shift_date,
    i.shift_type_norm                 AS shift_type,
    i.batch_id_norm                   AS batch_nbr,
    i.after_v:reason::string           AS failure_reason,
    regexp_replace(cast(i.after_v:to AS STRING), '([^,;\\s@]+)@', '***@') AS recipients,
    i.after_v:trigger::string          AS trigger_type,
    substring(regexp_replace(i.after_v:subject::string, '([^,;\\s@]+)@', '***@'), 1, 200) AS subject,
    current_timestamp()                AS detected_at
FROM mq_gmdf_dev.oil_obs.v_ish_bronze i
WHERE i.entity_type = 'HandoverEmail'
  AND i.after_v:sent::boolean = false
  AND i.is_email_disabled_gate = false;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- handover_delivery_rate — 7-day rolling, single row.
-- Daily rates are unusable: ~2 attempts/day means one failure swings the rate to 33% or 100%.
-- Baseline established 2026-08-20: 15 of 162 attempts failed since Jun 19 = 9.3%.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.handover_delivery_rate AS
SELECT
    count_if(after_v:sent::boolean = true)  AS sent_ok,
    count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false) AS failed,
    count_if(is_email_disabled_gate)        AS gated,
    round(count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false) * 100.0
          / nullif(count_if(after_v:sent::boolean = true)
                   + count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false), 0), 1)
      AS failure_pct_7d,
    max(ts) AS last_attempt
FROM mq_gmdf_dev.oil_obs.v_ish_bronze
WHERE entity_type = 'HandoverEmail'
  AND ts >= current_timestamp() - INTERVAL 7 DAYS;

num_affected_rows,num_inserted_rows


In [ ]:
%sql
-- handover_delivery_rate_findings — single-row when rate exceeds threshold.
-- Distinct from handover_delivery (per-failure, keyed on ish_row_id) — this is the
-- DETERIORATION signal. The rate catches worsening; the per-failure check catches any
-- single unacknowledged recurrence. Both are needed (md §8.4).
-- Literal string key: one condition, one incident, ever. detection_count carries persistence.
-- Floor of 10 attempts because a 7-day window at ~2 attempts/day swings wildly on small
-- numbers (daily rates hit 33% and 100% on a single failure).
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.handover_delivery_rate_findings AS
SELECT
    failure_pct_7d, sent_ok, failed, last_attempt,
    'handover_delivery_rate_7d' AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.handover_delivery_rate
WHERE failure_pct_7d > 20 AND (sent_ok + failed) >= 10;